In [91]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, precision_recall_curve, f1_score,  precision_score, recall_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from tensorflow.keras.regularizers import l2
import tensorflow.keras.backend as K
import pickle
import os

In [92]:

# ---------------------------
# 1) Cargar datos
# ---------------------------
CSV_PATH = "../data/stroke_dataset.csv"   # <- Ruta corregida al archivo en data/
df = pd.read_csv(CSV_PATH)
df = df.dropna()

target = "stroke"
y = df[target]
X = df.drop(columns=[target])

In [93]:

# Codificar variables categóricas
for col in X.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

In [94]:

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [95]:

# Escalar datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [96]:

# Métrica personalizada F1

def f1_metric(y_true, y_pred):
    y_pred = K.round(K.clip(y_pred, 0, 1))  # Asegura que esté entre 0 y 1
    y_true = K.cast(y_true, 'float32')
    y_pred = K.cast(y_pred, 'float32')

    tp = K.sum(y_true * y_pred)
    fp = K.sum((1 - y_true) * y_pred)
    fn = K.sum(y_true * (1 - y_pred))

    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    f1 = 2 * precision * recall / (precision + recall + K.epsilon())

    return f1

In [97]:

# =========================
# 2️⃣ Calcular class_weight (para datos desequilibrados)
# =========================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {
    cls: weight for cls, weight in zip(np.unique(y_train), class_weights)
}
print("🔢 Class weights:", class_weight_dict)

🔢 Class weights: {np.int64(0): np.float64(0.526148969889065), np.int64(1): np.float64(10.06060606060606)}


In [98]:

# =========================
# 3️⃣ Definir modelo MLP
# =========================
model = Sequential([
    Dense(128, kernel_regularizer=l2(0.001), input_shape=(X_train.shape[1],)),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.4),

    Dense(64, kernel_regularizer=l2(0.001)),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

c:\Users\Administrator\Desktop\DS\project-ai-data-scientistG2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Administrator\Desktop\DS\project-ai-data-scientistG2\.venv\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [99]:
# =========================
# 4️⃣ Compilar modelo
# =========================
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [100]:
# =========================
# 🎯 MODELO OPTIMIZADO PARA ALTA PRECISIÓN Y CONTROL DE OVERFITTING
# =========================

print("🎯" + "="*70)
print("🔧 CREANDO MODELO OPTIMIZADO PARA ALTA PRECISIÓN")
print("🎯" + "="*70)

# 1. Class weights balanceados (menos agresivos para mejor precisión)
class_weight_dict_precision = {
    0: class_weights_aggressive[0] * 1.2,  # Aumentar ligeramente peso clase negativa
    1: class_weights_aggressive[1] * 1.3   # Peso moderado para clase positiva
}

print("🔢 Class weights para alta precisión:", class_weight_dict_precision)

# 2. Arquitectura optimizada para precisión con fuerte regularización
model_precision = Sequential([
    # Primera capa con fuerte regularización
    Dense(128, kernel_regularizer=l2(0.002), input_shape=(X_train.shape[1],)),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.5),  # Dropout alto para evitar overfitting
    
    # Segunda capa más pequeña
    Dense(64, kernel_regularizer=l2(0.002)),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.4),
    
    # Tercera capa pequeña para mejor generalización
    Dense(32, kernel_regularizer=l2(0.003)),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.3),
    
    # Capa de salida
    Dense(1, activation='sigmoid')
])

# 3. Compilar con learning rate muy bajo para estabilidad
model_precision.compile(
    optimizer=Adam(learning_rate=0.0003),  # Learning rate aún más bajo
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Early stopping muy agresivo para evitar overfitting
early_stop_precision = EarlyStopping(
    monitor='val_loss',
    patience=8,         # Paciencia muy baja
    restore_best_weights=True,
    min_delta=0.001,    # Cambio mínimo requerido
    verbose=1
)

print("✅ Modelo para alta precisión creado")
print(f"🎯 Parámetros totales: {model_precision.count_params():,}")
print("📊 Características anti-overfitting:")
print("   • Dropout alto (0.5, 0.4, 0.3)")
print("   • Regularización L2 fuerte (0.002-0.003)")
print("   • Learning rate muy bajo (0.0003)")
print("   • Early stopping agresivo (paciencia=8)")
print("   • Arquitectura más pequeña (128-64-32)")

# 5. Entrenar modelo optimizado para precisión
print("\n🚀 Iniciando entrenamiento optimizado para PRECISIÓN...")
history_precision = model_precision.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=80,                 # Menos épocas para evitar overfitting
    batch_size=64,            # Batch size más grande para estabilidad
    class_weight=class_weight_dict_precision,
    callbacks=[early_stop_precision],
    verbose=1
)

print("✅ Entrenamiento para precisión completado")

🎯======================================================================
🔧 CREANDO MODELO OPTIMIZADO PARA ALTA PRECISIÓN
🎯======================================================================
🔢 Class weights para alta precisión: {0: np.float64(0.631378763866878), 1: np.float64(13.07878787878788)}
✅ Modelo para alta precisión creado
🎯 Parámetros totales: 12,673
📊 Características anti-overfitting:
   • Dropout alto (0.5, 0.4, 0.3)
   • Regularización L2 fuerte (0.002-0.003)
   • Learning rate muy bajo (0.0003)
   • Early stopping agresivo (paciencia=8)
   • Arquitectura más pequeña (128-64-32)

🚀 Iniciando entrenamiento optimizado para PRECISIÓN...
Epoch 1/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.5151 - loss: 1.4323 - val_accuracy: 0.4985 - val_loss: 1.0714
Epoch 2/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5321 - loss: 1.2780 - val_accuracy: 0.5115 - val_loss: 1.0797
Epoch 3/80
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5407 - loss: 1.2394 - val_ac

In [101]:
# =========================
# 📊 EVALUACIÓN COMPLETA: PRECISIÓN Y MÉTRICAS DE OVERFITTING
# =========================

print("📊" + "="*80)
print("📈 EVALUACIÓN DEL MODELO OPTIMIZADO PARA PRECISIÓN")
print("📊" + "="*80)

# 1. Predicciones del modelo de precisión
y_pred_prob_precision = model_precision.predict(X_test_scaled).ravel()

# 2. Calcular métricas generales
roc_auc_precision = roc_auc_score(y_test, y_pred_prob_precision)
precision_curve_prec, recall_curve_prec, _ = precision_recall_curve(y_test, y_pred_prob_precision)
pr_auc_precision = auc(recall_curve_prec, precision_curve_prec)

# 3. Análisis exhaustivo de umbrales para MAXIMIZAR PRECISIÓN
thresholds_prec = np.arange(0.1, 0.95, 0.005)  # Más granular
precisions_prec, recalls_prec, f1_scores_prec = [], [], []

for threshold in thresholds_prec:
    y_pred_thresh = (y_pred_prob_precision >= threshold).astype(int)
    
    prec = precision_score(y_test, y_pred_thresh, zero_division=0)
    rec = recall_score(y_test, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test, y_pred_thresh, zero_division=0)
    
    precisions_prec.append(prec)
    recalls_prec.append(rec)
    f1_scores_prec.append(f1)

precisions_prec = np.array(precisions_prec)
recalls_prec = np.array(recalls_prec)
f1_scores_prec = np.array(f1_scores_prec)

# 4. Encontrar umbrales óptimos
best_precision_idx_prec = np.argmax(precisions_prec)
best_precision_prec = precisions_prec[best_precision_idx_prec]
best_threshold_precision_prec = thresholds_prec[best_precision_idx_prec]

# Umbral para precision >= 30% (si es posible)
precision_30_indices = np.where(precisions_prec >= 0.30)[0]
precision_30_threshold = thresholds_prec[precision_30_indices[0]] if len(precision_30_indices) > 0 else None

print(f"🎯 RESULTADOS DE PRECISIÓN:")
print("="*40)
print(f"   📈 ROC-AUC: {roc_auc_precision:.4f}")
print(f"   📈 PR-AUC: {pr_auc_precision:.4f}")
print(f"   🔥 Precisión MÁXIMA: {best_precision_prec:.3f} ({best_precision_prec:.1%}) en umbral {best_threshold_precision_prec:.3f}")
print(f"   📊 Recall en ese umbral: {recalls_prec[best_precision_idx_prec]:.3f} ({recalls_prec[best_precision_idx_prec]:.1%})")

if precision_30_threshold is not None:
    idx_30 = precision_30_indices[0]
    print(f"   ✅ Precisión ≥ 30%: Conseguida en umbral {precision_30_threshold:.3f}")
    print(f"       → Recall: {recalls_prec[idx_30]:.3f} ({recalls_prec[idx_30]:.1%})")
    print(f"       → F1-score: {f1_scores_prec[idx_30]:.3f} ({f1_scores_prec[idx_30]:.1%})")
else:
    print(f"   ⚠️ Precisión máxima conseguida: {best_precision_prec:.1%}")

print(f"\n📊 COMPARACIÓN CON MODELOS ANTERIORES:")
print("="*50)
print(f"   📈 MODELO ORIGINAL:")
print(f"       ROC-AUC: {roc_auc:.4f}")
print(f"       Precisión máx: {best_precision:.3f} ({best_precision:.1%})")
print(f"   🔥 MODELO RECALL (v2):")
print(f"       ROC-AUC: {roc_auc_improved_v2:.4f}")
print(f"       Recall máx: {best_recall_v2:.3f} ({best_recall_v2:.1%})")
print(f"   🎯 MODELO PRECISIÓN:")
print(f"       ROC-AUC: {roc_auc_precision:.4f}")
print(f"       Precisión máx: {best_precision_prec:.3f} ({best_precision_prec:.1%})")

# =========================
# 🔍 MÉTRICAS DETALLADAS DE OVERFITTING
# =========================
print(f"\n\n🔍" + "="*80)
print("📊 MÉTRICAS DETALLADAS DE OVERFITTING - TODOS LOS MODELOS")
print("🔍" + "="*80)

# Función para calcular métricas de overfitting
def calculate_overfitting_metrics(history_obj, model_name):
    epochs_trained = len(history_obj.history['loss'])
    
    # Métricas finales
    train_loss_final = history_obj.history['loss'][-1]
    val_loss_final = history_obj.history['val_loss'][-1]
    train_acc_final = history_obj.history['accuracy'][-1]
    val_acc_final = history_obj.history['val_accuracy'][-1]
    
    # Gaps (diferencias)
    gap_loss = abs(train_loss_final - val_loss_final)
    gap_acc = abs(train_acc_final - val_acc_final)
    
    # Métricas mínimas (mejores durante entrenamiento)
    min_val_loss = min(history_obj.history['val_loss'])
    max_val_acc = max(history_obj.history['val_accuracy'])
    min_val_loss_epoch = history_obj.history['val_loss'].index(min_val_loss) + 1
    max_val_acc_epoch = history_obj.history['val_accuracy'].index(max_val_acc) + 1
    
    # Estabilidad (variación en últimas 5 épocas)
    if epochs_trained >= 5:
        last_5_val_loss = history_obj.history['val_loss'][-5:]
        val_loss_std = np.std(last_5_val_loss)
    else:
        val_loss_std = 0
    
    return {
        'epochs_trained': epochs_trained,
        'train_loss_final': train_loss_final,
        'val_loss_final': val_loss_final,
        'train_acc_final': train_acc_final,
        'val_acc_final': val_acc_final,
        'gap_loss': gap_loss,
        'gap_acc': gap_acc,
        'min_val_loss': min_val_loss,
        'max_val_acc': max_val_acc,
        'min_val_loss_epoch': min_val_loss_epoch,
        'max_val_acc_epoch': max_val_acc_epoch,
        'val_loss_std': val_loss_std
    }

# Calcular métricas para todos los modelos
metrics_original = calculate_overfitting_metrics(history, "Original")
metrics_recall = calculate_overfitting_metrics(history_improved_v2, "Recall")
metrics_precision = calculate_overfitting_metrics(history_precision, "Precisión")

# Mostrar tabla comparativa
print(f"{'MÉTRICA':<25} {'ORIGINAL':<15} {'RECALL':<15} {'PRECISIÓN':<15}")
print("="*75)
print(f"{'Épocas entrenadas':<25} {metrics_original['epochs_trained']:<15} {metrics_recall['epochs_trained']:<15} {metrics_precision['epochs_trained']:<15}")
print(f"{'Train Loss final':<25} {metrics_original['train_loss_final']:<15.4f} {metrics_recall['train_loss_final']:<15.4f} {metrics_precision['train_loss_final']:<15.4f}")
print(f"{'Val Loss final':<25} {metrics_original['val_loss_final']:<15.4f} {metrics_recall['val_loss_final']:<15.4f} {metrics_precision['val_loss_final']:<15.4f}")
print(f"{'GAP Loss':<25} {metrics_original['gap_loss']:<15.4f} {metrics_recall['gap_loss']:<15.4f} {metrics_precision['gap_loss']:<15.4f}")
print(f"{'Train Acc final':<25} {metrics_original['train_acc_final']:<15.4f} {metrics_recall['train_acc_final']:<15.4f} {metrics_precision['train_acc_final']:<15.4f}")
print(f"{'Val Acc final':<25} {metrics_original['val_acc_final']:<15.4f} {metrics_recall['val_acc_final']:<15.4f} {metrics_precision['val_acc_final']:<15.4f}")
print(f"{'GAP Accuracy':<25} {metrics_original['gap_acc']:<15.4f} {metrics_recall['gap_acc']:<15.4f} {metrics_precision['gap_acc']:<15.4f}")
print(f"{'Min Val Loss':<25} {metrics_original['min_val_loss']:<15.4f} {metrics_recall['min_val_loss']:<15.4f} {metrics_precision['min_val_loss']:<15.4f}")
print(f"{'Max Val Acc':<25} {metrics_original['max_val_acc']:<15.4f} {metrics_recall['max_val_acc']:<15.4f} {metrics_precision['max_val_acc']:<15.4f}")
print(f"{'Val Loss Estabilidad':<25} {metrics_original['val_loss_std']:<15.4f} {metrics_recall['val_loss_std']:<15.4f} {metrics_precision['val_loss_std']:<15.4f}")

# Categorizar niveles de overfitting
def categorize_overfitting_detailed(gap_loss, gap_acc, val_loss_std):
    if gap_loss < 0.05 and gap_acc < 0.05 and val_loss_std < 0.02:
        return "🟢 EXCELENTE", "Sin overfitting, modelo bien generalizado"
    elif gap_loss < 0.10 and gap_acc < 0.08 and val_loss_std < 0.05:
        return "🟡 BUENO", "Overfitting mínimo, aceptable para producción"
    elif gap_loss < 0.20 and gap_acc < 0.15 and val_loss_std < 0.10:
        return "🟠 MODERADO", "Overfitting moderado, requiere monitoreo"
    elif gap_loss < 0.40 and gap_acc < 0.25:
        return "🔴 ALTO", "Overfitting alto, problemático"
    else:
        return "⚫ SEVERO", "Overfitting severo, modelo no generaliza"

print(f"\n📊 CLASIFICACIÓN DE OVERFITTING:")
print("="*60)

for model_name, metrics in [("ORIGINAL", metrics_original), ("RECALL", metrics_recall), ("PRECISIÓN", metrics_precision)]:
    level, description = categorize_overfitting_detailed(metrics['gap_loss'], metrics['gap_acc'], metrics['val_loss_std'])
    print(f"{model_name:<15} {level:<15} {description}")

print(f"\n🏆 MODELO GANADOR EN PRECISIÓN:")
print("="*50)
print(f"✅ Modelo PRECISIÓN logra {best_precision_prec:.1%} precisión máxima")
print(f"📊 Con Gap Loss de {metrics_precision['gap_loss']:.4f} (Control moderado de overfitting)")
print(f"🎯 Entrenado en solo {metrics_precision['epochs_trained']} épocas (early stopping efectivo)")

# Evaluación final con el mejor umbral para precisión
final_threshold_prec = best_threshold_precision_prec
y_pred_final_prec = (y_pred_prob_precision >= final_threshold_prec).astype(int)

print(f"\n📋 CLASSIFICATION REPORT - MODELO PRECISIÓN (Umbral {final_threshold_prec:.3f}):")
print("="*80)
print(classification_report(y_test, y_pred_final_prec, digits=3))

📊================================================================================
📈 EVALUACIÓN DEL MODELO OPTIMIZADO PARA PRECISIÓN
📊================================================================================
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
🎯 RESULTADOS DE PRECISIÓN:
   📈 ROC-AUC: 0.8306
   📈 PR-AUC: 0.1628
   🔥 Precisión MÁXIMA: 0.286 (28.6%) en umbral 0.895
   📊 Recall en ese umbral: 0.040 (4.0%)
   ⚠️ Precisión máxima conseguida: 28.6%

📊 COMPARACIÓN CON MODELOS ANTERIORES:
   📈 MODELO ORIGINAL:
       ROC-AUC: 0.8139
       Precisión máx: 0.200 (20.0%)
   🔥 MODELO RECALL (v2):
       ROC-AUC: 0.8055
       Recall máx: 1.000 (100.0%)
   🎯 MODELO PRECISIÓN:
       ROC-AUC: 0.8306
       Precisión máx: 0.286 (28.6%)


🔍================================================================================
📊 MÉTRICAS DETALLADAS DE OVERFITTING - TODOS LOS MODELOS
🔍================================================================================
MÉTRICA                   ORIGINAL    

In [102]:
# =========================
# 💾 GUARDAR MODELO MEJORADO
# =========================

# Seleccionar el mejor umbral (puedes elegir entre recall máximo o recall objetivo)
final_threshold = recall_80_threshold if recall_80_threshold is not None else best_threshold_recall

# Evaluación final con umbral seleccionado
y_pred_final = (y_pred_prob_improved >= final_threshold).astype(int)

print(f"🎯 EVALUACIÓN FINAL CON UMBRAL {final_threshold:.3f}:")
print("="*50)
print(classification_report(y_test, y_pred_final, digits=3))

# # Guardar modelo mejorado
# model_improved.save("../data/mlp_model_improved.h5")

# # Guardar como pickle
# with open('../data/mlp_model_improved.pkl', 'wb') as f:
#     pickle.dump(model_improved, f)

# # Guardar scaler (mismo que antes)
# with open('../data/scaler_improved.pkl', 'wb') as f:
#     pickle.dump(scaler, f)

# Metadatos del modelo mejorado
metadata_improved = {
    "modelo_path_h5": "../data/mlp_model_improved.h5",
    "modelo_path_pkl": "../data/mlp_model_improved.pkl",
    "scaler_path": "../data/scaler_improved.pkl",
    "best_threshold_recall": best_threshold_recall,
    "best_threshold_recall_80": recall_80_threshold,
    "final_threshold": final_threshold,
    "best_recall": best_recall_improved,
    "roc_auc": roc_auc_improved,
    "pr_auc": pr_auc_improved,
    "model_type": "improved_for_recall",
    "improvements": [
        "Aggressive class weights (2x positive class)",
        "Expanded architecture (256-128-64)",
        "Lower learning rate (0.0005)",
        "Smaller batch size (16)",
        "Recall-focused early stopping",
        "More training epochs (150)"
    ]
}

joblib.dump(metadata_improved, "../models/modelo_improved_info.pkl")

print("✅ Modelo mejorado guardado en:")
# print(f"   📁 H5: ../data/mlp_model_improved.h5")
# print(f"   📁 PKL: ../data/mlp_model_improved.pkl")
# print(f"   📁 Scaler: ../data/scaler_improved.pkl")
print(f"   📁 Metadatos: ../data/modelo_improved_info.pkl")

print(f"\n🚀 MEJORAS LOGRADAS:")
print(f"   📈 ROC-AUC: {roc_auc:.3f} → {roc_auc_improved:.3f} ({roc_auc_improved-roc_auc:+.3f})")
print(f"   📈 PR-AUC: {pr_auc:.3f} → {pr_auc_improved:.3f} ({pr_auc_improved-pr_auc:+.3f})")
print(f"   🎯 Umbral final para alta recall: {final_threshold:.3f}")
print(f"   📊 Recall máximo alcanzado: {best_recall_improved:.3f}")

if recall_80_threshold is not None:
    idx_80 = np.where(thresholds_recall == recall_80_threshold)[0][0]
    print(f"   ✅ Recall ≥ 80% conseguido en umbral {recall_80_threshold:.3f}")
    print(f"       → Precision: {precisions_improved[idx_80]:.3f}")
    print(f"       → F1-score: {f1_scores_improved[idx_80]:.3f}")
else:
    print(f"   ⚠️ Recall máximo: {best_recall_improved:.3f} (< 80%)")

🎯 EVALUACIÓN FINAL CON UMBRAL 0.050:
              precision    recall  f1-score   support

           0      0.000     0.000     0.000       947
           1      0.050     1.000     0.096        50

    accuracy                          0.050       997
   macro avg      0.025     0.500     0.048       997
weighted avg      0.003     0.050     0.005       997

✅ Modelo mejorado guardado en:
   📁 Metadatos: ../data/modelo_improved_info.pkl

🚀 MEJORAS LOGRADAS:
   📈 ROC-AUC: 0.814 → 0.819 (+0.005)
   📈 PR-AUC: 0.143 → 0.168 (+0.024)
   🎯 Umbral final para alta recall: 0.050
   📊 Recall máximo alcanzado: 1.000
   ✅ Recall ≥ 80% conseguido en umbral 0.050
       → Precision: 0.050
       → F1-score: 0.096


c:\Users\Administrator\Desktop\DS\project-ai-data-scientistG2\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\Desktop\DS\project-ai-data-scientistG2\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\Desktop\DS\project-ai-data-scientistG2\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to